In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path 

Lectura de Datos

In [2]:
url_cgsse = 'https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/GEIH2025/cgsse.CSV'
url_ocupados = 'https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/GEIH2025/Ocupados.CSV'
url_ft = 'https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/GEIH2025/Fuerza%20de%20trabajo.CSV'
url_desocupados = 'https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/GEIH2025/No%20ocupados.CSV'

# CGSSE
df_cgsse = pd.read_csv(url_cgsse, encoding='latin-1', sep=';', 
                       low_memory=False)

#Ocupados
df_ocupados=pd.read_csv(url_ocupados,encoding='latin-1',sep=';',
                        low_memory=False)

#Fuerza de trabajo
df_ft=pd.read_csv(url_ft,
                  encoding='latin-1',sep=';',low_memory=False)

#No ocupados
df_desocupados=pd.read_csv(url_desocupados,encoding='latin-1',
                           sep=';',low_memory=False)

Para el total nacional calcule :
 
-población total
-población en edad de trabajar
-población que no está en edad de trabajar
-población económicamente activa
-población económicamente inactiva
-tasa de ocupación
-tasa de desempleo



- 1. Poblacion total

In [3]:
Población_total = df_cgsse['FEX_C18'].sum()
Población_total

np.float64(52045001.000009045)

Lo que nos quiere decir (np.float64(52045001.000009045)) es que la poblacion total es de 52.045.001

- Población en edad de trabajar

In [4]:
PET = df_cgsse[df_cgsse['P6040'] >= 15]['FEX_C18'].sum()
PET

np.float64(40653416.00000409)

Lo que nos quiere decir (np.float64(40653416.00000409)) es que hay 40.653.416 en edad de trabajar

- No edad de trabajar

In [5]:
NOET = df_cgsse[df_cgsse['P6040'] < 15]['FEX_C18'].sum()
NOET

np.float64(11391585.000004958)

Lo que nos quiere decir np.float64(11391585.000004958) es que hay 11.391.585 en no edad de trabajar

-PEA

In [7]:
PEA = df_ft[(df_ft['FT']==1)]['FEX_C18'].sum()
PEA

np.float64(25974846.22356785)

Lo que nos quiere decir np.float64(25974846.22356785) es que hay 25.974.846 personas economicamente activas

PEI

In [8]:
PEI = df_ft[(df_ft['FFT']==1)]['FEX_C18'].sum()
PEI

np.float64(14678569.776436238)

Lo que nos quiere decir np.float64(14678569.776436238) es que hay 14.678.569 personas economicamente inactivas

-TO

In [9]:
Ocupados = df_ocupados['FEX_C18'].sum()
Ocupados

np.float64(23752952.660255846)

In [10]:
Tasa_de_ocupación = (Ocupados/PET)*100
Tasa_de_ocupación

np.float64(58.427937913639184)

Lo que nos quiere decir np.float64(58.427937913639184) es que la tasa de ocupados es de un 58.42%

-TD

In [11]:
Desocupados = df_desocupados[df_desocupados['DSI']==1]['FEX_C18'].sum()
Desocupados

np.float64(2221893.563312005)

In [12]:
Tasa_de_desempleo = (Desocupados/PEA)*100
Tasa_de_desempleo

np.float64(8.554020086155528)

Lo que nos quiere decir np.float64(8.554020086155528) es que la tasa de desempleo es del 8.55%

2.Calcule las tasas de desempleo para hombres y mujeres en Risaralda, Caldas , Tolima y Quindío. ¿Cuál es el departamento con mayor y menor tasa de desempleo de los 4 ? 


In [15]:
region_cod = [17, 63, 66, 73]

desocupados_region = df_desocupados[df_desocupados['DPTO'].isin(region_cod)].copy()
ocupados_region = df_ocupados[df_ocupados['DPTO'].isin(region_cod)].copy()

desoc_merge = pd.merge(
    desocupados_region[desocupados_region["DSI"] == 1],
    df_cgsse[['DIRECTORIO','SECUENCIA_P','ORDEN','P3271']],
    on=['DIRECTORIO','SECUENCIA_P','ORDEN'],
    how='left'
)

ocup_merge = pd.merge(
    ocupados_region,
    df_cgsse[['DIRECTORIO','SECUENCIA_P','ORDEN','P3271']],
    on=['DIRECTORIO','SECUENCIA_P','ORDEN'],
    how='left'
)

total_desocupados = desoc_merge.groupby(['DPTO','P3271'])['FEX_C18'].sum().reset_index(name='Total_desocupados')
total_ocupados = ocup_merge.groupby(['DPTO','P3271'])['FEX_C18'].sum().reset_index(name='Total_ocupados')

tabla_pea = pd.merge(total_ocupados, total_desocupados, on=['DPTO','P3271'], how='outer').fillna(0)

tabla_pea['PEA_total'] = tabla_pea['Total_ocupados'] + tabla_pea['Total_desocupados']
tabla_pea['Tasa_desempleo'] = (tabla_pea['Total_desocupados'] / tabla_pea['PEA_total']) * 100

deptos = {
    17: 'Caldas',
    63: 'Quindio',
    66: 'Risaralda',
    73: 'Tolima'
}

tabla_pea['Depto'] = tabla_pea['DPTO'].map(deptos)
tabla_pea['Genero'] = tabla_pea['P3271'].map({1:'Hombre', 2:'Mujer'})

resultado = tabla_pea[['Depto', 'Genero', 'Tasa_desempleo']].sort_values(['Depto','Genero'])

print(resultado)

       Depto  Genero  Tasa_desempleo
0     Caldas  Hombre        5.583183
1     Caldas   Mujer        7.677962
2    Quindio  Hombre       10.233032
3    Quindio   Mujer        9.084532
4  Risaralda  Hombre        6.074260
5  Risaralda   Mujer       10.669927
6     Tolima  Hombre        7.220583
7     Tolima   Mujer       19.202747


In [17]:
td_mayor = tabla_pea.loc[tabla_pea.groupby("Genero")["Tasa_desempleo"].idxmax()]
td_menor = tabla_pea.loc[tabla_pea.groupby("Genero")["Tasa_desempleo"].idxmin()]

print("\nMayor tasa de desempleo por genero")
print(td_mayor[['Depto', 'Genero', 'Tasa_desempleo']])

print("\nMenor tasa de desempleo por genero")
print(td_menor[['Depto', 'Genero', 'Tasa_desempleo']])


Mayor tasa de desempleo por genero
     Depto  Genero  Tasa_desempleo
2  Quindio  Hombre       10.233032
7   Tolima   Mujer       19.202747

Menor tasa de desempleo por genero
    Depto  Genero  Tasa_desempleo
0  Caldas  Hombre        5.583183
1  Caldas   Mujer        7.677962


Lo que nos muestre es que la mayor tasa de desempleo por parte de hombres es del quindio con un resultado de 10.23% caso de las mujeres en tolima con un 19.20% y en el caso de menor desempleo ambos son de caldas con un porcentaje del 5.5 en hombres y un 7.67 en mujeres


Tomando en consideración únicamente los siguientes grupos de edad: 

18 a 24
24 a 30
30 a 40
40 a 50 
60 en adelante 

Para Risaralda, Caldas , Tolima y Quindío, cual es el grupo de edad con mayor y menor tasa de desempleo (indicar en cada departamento el grupo de de edad  con mayor y menor tasa de desempleo) ? 


In [18]:
region_cod = [17, 63, 66, 73]

def grupo_etario(x):
    if 18 <= x < 24:
        return "18-24"
    elif 24 <= x < 30:
        return "24-30"
    elif 30 <= x < 40:
        return "30-40"
    elif 40 <= x < 50:
        return "40-50"
    elif x >= 60:
        return "60 o más"
    else:
        return "Otro rango"

df_cgsse["grupo_edad"] = df_cgsse["P6040"].apply(grupo_etario)

desocupados_region = df_desocupados[df_desocupados["DPTO"].isin(region_cod)].copy()

des_edad_stats = pd.merge(
    desocupados_region[desocupados_region["DSI"] == 1],
    df_cgsse[["DIRECTORIO","SECUENCIA_P","ORDEN","grupo_edad"]],
    on=["DIRECTORIO","SECUENCIA_P","ORDEN"],
    how="left"
).groupby(["DPTO","grupo_edad"])["FEX_C18"].sum().reset_index(name="Total_desocupados")

ocupados_region = df_ocupados[df_ocupados["DPTO"].isin(region_cod)].copy()

ocu_edad_stats = pd.merge(
    ocupados_region,
    df_cgsse[["DIRECTORIO","SECUENCIA_P","ORDEN","grupo_edad"]],
    on=["DIRECTORIO","SECUENCIA_P","ORDEN"],
    how="left"
).groupby(["DPTO","grupo_edad"])["FEX_C18"].sum().reset_index(name="Total_ocupados")

tabla_pea = pd.merge(
    ocu_edad_stats,
    des_edad_stats,
    on=["DPTO","grupo_edad"],
    how="outer"
).fillna(0)

tabla_pea["PEA_total"] = tabla_pea["Total_ocupados"] + tabla_pea["Total_desocupados"]
tabla_pea["Tasa_desempleo"] = (tabla_pea["Total_desocupados"] / tabla_pea["PEA_total"]) * 100

rangos_validos = ["18-24", "24-30", "30-40", "40-50", "60 o más"]
tabla_pea = tabla_pea[tabla_pea["grupo_edad"].isin(rangos_validos)].copy()

deptos = {
    17: "Caldas",
    63: "Quindio",
    66: "Risaralda",
    73: "Tolima"
}

tabla_pea["Depto"] = tabla_pea["DPTO"].map(deptos)

mayor_td = tabla_pea.loc[tabla_pea.groupby("Depto")["Tasa_desempleo"].idxmax()]
menor_td = tabla_pea.loc[tabla_pea.groupby("Depto")["Tasa_desempleo"].idxmin()]

print("\nMayor tasa de desempleo por departamento:")
print(mayor_td[["Depto", "grupo_edad", "Tasa_desempleo"]])

print("\nMenor tasa de desempleo por departamento:")
print(menor_td[["Depto", "grupo_edad", "Tasa_desempleo"]])


Mayor tasa de desempleo por departamento:
        Depto grupo_edad  Tasa_desempleo
0      Caldas      18-24        9.972584
7     Quindio      24-30       14.237924
12  Risaralda      18-24       15.173556
18     Tolima      18-24       19.348020

Menor tasa de desempleo por departamento:
        Depto grupo_edad  Tasa_desempleo
4      Caldas   60 o más        4.060989
9     Quindio      40-50        6.506359
16  Risaralda   60 o más        3.639861
22     Tolima   60 o más        9.458632


los resultados muestran que las tasas de desempleo más altas se concentran principalmente en los grupos de edad más jóvenes, especialmente entre 18 y 24 años, lo que evidencia mayores dificultades de inserción laboral para la población que apenas inicia su vida laboral. En este grupo destacan Tolima (19,35%), Risaralda (15,17%) y Caldas (9,97%), mientras que en Quindío la mayor tasa se presenta en el grupo 24-30 años (14,24%).

Por el contrario, las tasas de desempleo más bajas se registran en los grupos de mayor edad, especialmente en personas de 60 años o más, lo que sugiere que quienes permanecen en el mercado laboral a estas edades tienden a tener mayor estabilidad o menor participación en la búsqueda de empleo. En este caso destacan Risaralda (3,64%), Caldas (4,06%) y Tolima (9,46%), mientras que en Quindío la menor tasa se presenta en el grupo 40-50 años (6,51%).

Entre Risaralda, Caldas , Tolima y Quindío, cuál es el departamento con menor tasa de desempleo de mujeres entre los 18 y los 25 años ? 



In [21]:
region_cod = [17, 63, 66, 73]

# 2. Base de desocupados con variables de sexo y edad
desocupados_region = pd.merge(
    df_desocupados[(df_desocupados["DSI"] == 1) & (df_desocupados["DPTO"].isin(region_cod))],
    df_cgsse[["DIRECTORIO","SECUENCIA_P","ORDEN","P3271","P6040"]],
    on=["DIRECTORIO","SECUENCIA_P","ORDEN"],
    how="left"
)

# 3. Base de ocupados con variables de sexo y edad
ocupados_region = pd.merge(
    df_ocupados[df_ocupados["DPTO"].isin(region_cod)],
    df_cgsse[["DIRECTORIO","SECUENCIA_P","ORDEN","P3271","P6040"]],
    on=["DIRECTORIO","SECUENCIA_P","ORDEN"],
    how="left"
)

# 4. Filtrar mujeres jóvenes (18 a 25 años)

# Desocupadas
desocupadas_jovenes = desocupados_region[
    (desocupados_region["P3271"] == 2) &
    (desocupados_region["P6040"].between(18,25))
].groupby("DPTO")["FEX_C18"].sum()

# Ocupadas
ocupadas_jovenes = ocupados_region[
    (ocupados_region["P3271"] == 2) &
    (ocupados_region["P6040"].between(18,25))
].groupby("DPTO")["FEX_C18"].sum()
# 5. Calcular población económicamente activa
pea_jovenes = ocupadas_jovenes.add(desocupadas_jovenes, fill_value=0)

tabla_resultado = pd.DataFrame({
    "PEA_total": pea_jovenes,
    "Desocupadas": desocupadas_jovenes
}).fillna(0)

tabla_resultado["Tasa_desempleo"] = (tabla_resultado["Desocupadas"] / tabla_resultado["PEA_total"]) * 100

# 6. Agregar nombres de departamentos
deptos = {
    17: "Caldas",
    63: "Quindio",
    66: "Risaralda",
    73: "Tolima"
}

tabla_resultado["Depto"] = tabla_resultado.index.map(deptos)

tabla_resultado = tabla_resultado.sort_values(by="Tasa_desempleo")

# Mostrar resultados
print("=== Tasa de Desempleo Mujeres 18-25 años (Región) ===")
print(tabla_resultado[["Depto","Tasa_desempleo"]])

=== Tasa de Desempleo Mujeres 18-25 años (Región) ===
          Depto  Tasa_desempleo
DPTO                           
63      Quindio        7.468211
17       Caldas       14.500102
73       Tolima       21.378071
66    Risaralda       22.992769


Los resultados muestran que la tasa de desempleo de mujeres entre 18 y 25 años presenta diferencias importantes entre los departamentos analizados. El menor nivel de desempleo juvenil femenino se registra en Quindío (7,47%), lo que sugiere una mayor inserción laboral de este grupo etario en comparación con los demás territorios.

Por su parte, Caldas presenta una tasa intermedia (14,50%), mientras que Tolima (21,38%) y Risaralda (22,99%) registran los niveles más altos de desempleo para mujeres jóvenes. En síntesis, el análisis evidencia que las mayores dificultades de acceso al mercado laboral para mujeres jóvenes se concentran en Tolima y Risaralda, mientras que Quindío muestra un comportamiento relativamente más favorable dentro de la región.

¿Cuántos hombres y mujeres hay en edad de trabajar en el quindio ? ¿Qué porcentaje de la población total representan ? 

In [22]:
# Filtrar información del departamento del Quindío
base_quindio = df_cgsse[df_cgsse["DPTO"] == 63].copy()

# Identificar población en edad de trabajar (15 años o más)
base_quindio["edad_trabajo"] = base_quindio["P6040"] >= 15

# Población total estimada
pob_total = base_quindio["FEX_C18"].sum()

# Cálculo de PET por sexo
pet_sexo = base_quindio[base_quindio["edad_trabajo"] == True].groupby("P3271")["FEX_C18"].sum()

pet_hombres = pet_sexo.get(1, 0)
pet_mujeres = pet_sexo.get(2, 0)
pet_total = pet_hombres + pet_mujeres

# Porcentajes respecto a la población total
porc_hombres = (pet_hombres / pob_total) * 100
porc_mujeres = (pet_mujeres / pob_total) * 100
porc_pet_total = (pet_total / pob_total) * 100

print(f"Población total estimada en Quindío: {int(pob_total)}")
print(f"Hombres en edad de trabajar: {int(pet_hombres)} ({porc_hombres:.2f}% del total)")
print(f"Mujeres en edad de trabajar: {int(pet_mujeres)} ({porc_mujeres:.2f}% del total)")
print(f"Total población en edad de trabajar (PET): {int(pet_total)} ({porc_pet_total:.2f}% del total)")

Población total estimada en Quindío: 603068
Hombres en edad de trabajar: 247037 (40.96% del total)
Mujeres en edad de trabajar: 261387 (43.34% del total)
Total población en edad de trabajar (PET): 508425 (84.31% del total)


Los resultados muestran que la población total estimada del Quindío es de 603.068 personas, de las cuales 508.425 se encuentran en edad de trabajar (15 años o más), lo que representa el 84,31% de la población total del departamento. Esto indica que una gran parte de la población se encuentra potencialmente disponible para participar en el mercado laboral.

Al analizar la composición por sexo dentro de la población en edad de trabajar, se observa que las mujeres representan una proporción ligeramente mayor, con 261.387 personas (43,34% de la población total), mientras que los hombres suman 247.037 (40,96%). En síntesis, la estructura demográfica del Quindío muestra una amplia base de población en edad productiva y una leve predominancia femenina dentro de este grupo, lo cual puede tener implicaciones en la dinámica del mercado laboral del departamento.


¿Qué porcentaje de la población del Tolima está en edad de trabajar ? 


In [23]:
# 1. Filtrar información del departamento del Tolima
base_tolima = df_cgsse[df_cgsse["DPTO"] == 73].copy()

# 2. Crear indicador de población en edad de trabajar (15 años o más)
base_tolima["edad_trabajo"] = base_tolima["P6040"] >= 15

# 3. Calcular población total y población en edad de trabajar usando el factor de expansión
total_poblacion = base_tolima["FEX_C18"].sum()
pet_poblacion = base_tolima[base_tolima["edad_trabajo"] == True]["FEX_C18"].sum()

# 4. Calcular proporción de PET
porc_pet = (pet_poblacion / total_poblacion) * 100

print(f"Población total estimada en Tolima: {int(total_poblacion)}")
print(f"Población en edad de trabajar: {int(pet_poblacion)}")
print(f"Porcentaje de población en edad de trabajar: {porc_pet:.2f}%")

Población total estimada en Tolima: 1411965
Población en edad de trabajar: 1087359
Porcentaje de población en edad de trabajar: 77.01%


La población total estimada del Tolima es de 1.411.965 personas, de las cuales 1.087.359 se encuentran en edad de trabajar, lo que representa aproximadamente el 77,01% de la población total, evidenciando que una parte importante de la población se encuentra potencialmente disponible para participar en el mercado laboral.

¿ Qué porcentaje de la población de Risaralda es población económicamente inactiva ?

In [24]:
# Población económicamente inactiva en Risaralda
pei_risaralda = df_ft[(df_ft["FFT"] == 1) & (df_ft["DPTO"] == 66)]["FEX_C18"].sum()

# Población en edad de trabajar en Risaralda
pet_risaralda = df_cgsse[df_cgsse["DPTO"] == 66]["FEX_C18"].sum()

# Cálculo del porcentaje de inactivos
porc_pei = (pei_risaralda / pet_risaralda) * 100

print(f"Población en edad de trabajar en Risaralda: {int(pet_risaralda)}")
print(f"Población económicamente inactiva en Risaralda: {int(pei_risaralda)}")
print(f"Porcentaje de población inactiva: {porc_pei:.2f}%")

Población en edad de trabajar en Risaralda: 1025240
Población económicamente inactiva en Risaralda: 332743
Porcentaje de población inactiva: 32.46%


En Risaralda, la población en edad de trabajar asciende a 1.025.240 personas, de las cuales 332.743 son económicamente inactivas, lo que representa aproximadamente el 32,46% de la población en edad de trabajar.

¿Cuántas personas hay en el Quindío en condición de desempleo entre los 18 y los 30 años ? 



In [25]:
# 1. Unir base de desocupados con información de edad y departamento
base_desempleo = pd.merge(
    df_desocupados,
    df_cgsse[['DIRECTORIO','SECUENCIA_P','ORDEN','P6040','DPTO']],
    on=['DIRECTORIO','SECUENCIA_P','ORDEN'],
    how='left'
)

# 2. Filtrar desempleados jóvenes en Quindío (18 a 30 años)
desempleo_joven_quindio = base_desempleo[
    (base_desempleo['DPTO_y'] == 63) &
    (base_desempleo['DSI'] == 1) &
    (base_desempleo['P6040'].between(18,30))
]['FEX_C18'].sum()

print("Número de personas desempleadas entre 18 y 30 años en Quindío:", round(desempleo_joven_quindio))

Número de personas desempleadas entre 18 y 30 años en Quindío: 9480


En el Quindío se estiman 9.480 personas desempleadas entre 18 y 30 años, lo que evidencia la presencia de desempleo dentro de la población joven del departamento.